# Confronto metriche `*_dist_nll_corr` e `*_sep_corr` su N history.json

Imposta una lista di percorsi a file `history.json`, poi esegui tutte le celle.
Ogni subplot mostra una metrica confrontando tutti i path forniti.

In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for candidate in [base, *base.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Project root non trovato (attesi folder 'src' e 'data').")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dpo_config import Config
from src.dpo_data import pad_encode, read_fasta
from src.dpo_metrics import compute_sequence_nll
from src.dpo_plotting import (
    _build_binary_labels_and_scores,
    _compute_pr_curve,
    _compute_roc_curve,
)
from src.transformer import GPTTransformer

cfg = Config()
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

split_dir = project_root / "data/split_train_validation/split_fraction-0.15_add-vae-25-30-and-close-to-validation_nll-filtered_100"
dn_path = project_root / cfg.dn_path
pretrained_ckpt_path = project_root / cfg.pretrained_ckpt_path

if not dn_path.exists():
    raise FileNotFoundError(f"DN FASTA non trovato: {dn_path}")
if not pretrained_ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint pretrained non trovato: {pretrained_ckpt_path}")

train_good_path = split_dir / "DTp_train.fasta"
train_bad_path = split_dir / "DTm_train_after-nll-filter.fasta"
val_good_path = split_dir / "DTp_val_plus-vae25-30-and-close.fasta"
val_bad_path = split_dir / "DTm_val_plus-vae25-30-and-close.fasta"
val_1_good_path = split_dir / "DTp_vae25-30_plus-close-close.fasta"
val_1_bad_path = split_dir / "DTm_vae25-30_plus-close-close.fasta"

for p in [
    train_good_path,
    train_bad_path,
    val_good_path,
    val_bad_path,
    val_1_good_path,
    val_1_bad_path,
]:
    if not p.exists():
        raise FileNotFoundError(f"File split non trovato: {p}")

with dn_path.open("r", encoding="utf-8") as f:
    dn_lines = f.read().splitlines()[1::2]
chars = sorted(list(set("\n".join(dn_lines))))
pad_symbol = "?"
chars = chars + [pad_symbol]
stoi = {ch: i for i, ch in enumerate(chars)}

pad_token = stoi[pad_symbol]
vocab_size = len(stoi)


def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


train_good_sequences = read_fasta(train_good_path)
train_bad_sequences = read_fasta(train_bad_path)
val_good_sequences = read_fasta(val_good_path)
val_bad_sequences = read_fasta(val_bad_path)
val_1_good_sequences = read_fasta(val_1_good_path)
val_1_bad_sequences = read_fasta(val_1_bad_path)

train_good_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in train_good_sequences]
train_bad_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in train_bad_sequences]
val_good_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_good_sequences]
val_bad_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_bad_sequences]
val_1_good_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_1_good_sequences]
val_1_bad_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_1_bad_sequences]

print(f"Project root: {project_root}")
print(f"Device: {device}")
print("Dataset sizes (good/bad):")
print(f"  train: {len(train_good_data)} / {len(train_bad_data)}")
print(f"  val:   {len(val_good_data)} / {len(val_bad_data)}")
print(f"  val_1: {len(val_1_good_data)} / {len(val_1_bad_data)}")

In [ ]:
# Inserisci qui tutti i path che vuoi confrontare.
history_paths = [
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top5_bin/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top20_bin/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top500_bin/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin_refbin/history.json", 
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin_refbin_reciprocal/history.json", 
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs_bin_refbin_reciprocal/history.json"
    
    
]

if len(history_paths) == 0:
    raise ValueError("Inserisci almeno un percorso in history_paths.")

In [ ]:
def _parse_constant(value: str):
    # I file possono contenere NaN/Infinity non strettamente JSON standard.
    mapping = {"NaN": float("nan"), "Infinity": float("inf"), "-Infinity": float("-inf")}
    if value in mapping:
        return mapping[value]
    raise ValueError(f"Costante non supportata nel JSON: {value}")


def load_history(path_str: str) -> pd.DataFrame:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f, parse_constant=_parse_constant)

    required = [
        "iteration",
        "train_dist_nll_corr",
        "val_dist_nll_corr",
        "val_1_dist_nll_corr",
        "val_sep_corr",
        "val_1_sep_corr",
    ]
    missing = [k for k in required if k not in raw]
    if missing:
        raise KeyError(f"Nel file {path} mancano le chiavi: {missing}")

    min_len = min(len(raw[k]) for k in required)
    if min_len == 0:
        raise ValueError(f"Nessun dato utile in {path}")

    return pd.DataFrame({
        "iteration": raw["iteration"][:min_len],
        "train": raw["train_dist_nll_corr"][:min_len],
        "validation": raw["val_dist_nll_corr"][:min_len],
        "validation 1": raw["val_1_dist_nll_corr"][:min_len],
        "val sep": raw["val_sep_corr"][:min_len],
        "val 1 sep": raw["val_1_sep_corr"][:min_len],
    })


dataframes = [load_history(p) for p in history_paths]
[df.head(3) for df in dataframes]

In [ ]:
metric_map = [
    ("train", "train_dist_nll_corr"),
    ("validation", "val_dist_nll_corr"),
    ("validation 1", "val_1_dist_nll_corr"),
    ("val sep", "val_sep_corr"),
    ("val 1 sep", "val_1_sep_corr"),
]

n_metrics = len(metric_map)
n_cols = 3
n_rows = (n_metrics + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), sharex=False, sharey=False)
axes = axes.flatten()

for ax, (col, metric_name) in zip(axes, metric_map):
    for path_str, df in zip(history_paths, dataframes):
        run_label = str(Path(path_str).parent.name)
        ax.plot(df["iteration"], df[col], label=run_label, linewidth=2)

    ax.set_title(f"Metric: {metric_name}")
    ax.set_xlabel("iteration")
    ax.set_ylabel(metric_name)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

for ax in axes[n_metrics:]:
    ax.axis("off")

fig.suptitle("Confronto dei path per ciascuna metrica", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def build_model() -> GPTTransformer:
    model = GPTTransformer(
        vocab_size=vocab_size,
        embed_dim=cfg.n_embd,
        num_layers=cfg.n_layer,
        num_heads=cfg.n_head,
        mlp_ratio=4,
        dropout_p=0.1,
        pad_id=pad_token,
    )
    return model.to(device)


def resolve_history_path(path_str: str) -> Path:
    p = Path(path_str)
    if p.is_absolute():
        return p
    # I path nel notebook sono relativi alla cartella notebooks.
    return (project_root / "notebooks" / p).resolve()


def checkpoint_dir_from_history_path(history_path: Path) -> Path:
    rel = history_path.resolve().relative_to(project_root)
    if len(rel.parts) < 3 or rel.parts[0] != "images_dpo":
        raise ValueError(f"Path history non supportato per mapping checkpoint: {history_path}")
    return project_root / "checkpoints" / "checkpoints_dpo" / Path(*rel.parts[1:-1])


def list_checkpoint_iterations(ckpt_dir: Path) -> dict[int, Path]:
    pattern = re.compile(r"dpo_model_iter(\d+)\.pt$")
    found: dict[int, Path] = {}
    if not ckpt_dir.exists():
        return found
    for ckpt in ckpt_dir.glob("dpo_model_iter*.pt"):
        m = pattern.search(ckpt.name)
        if m is None:
            continue
        found[int(m.group(1))] = ckpt
    return dict(sorted(found.items(), key=lambda kv: kv[0]))


def _normalize_state_dict_keys(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    # Alcuni checkpoint salvati con torch.compile hanno il prefisso '_orig_mod.'.
    if all(k.startswith("_orig_mod.") for k in state_dict.keys()):
        return {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
    return state_dict


def compute_auroc_auprc_from_nll(good_nll: list[float], bad_nll: list[float]) -> tuple[float, float]:
    labels, scores = _build_binary_labels_and_scores(good_nll, bad_nll)
    _, _, auroc = _compute_roc_curve(labels, scores)
    _, _, auprc = _compute_pr_curve(labels, scores)
    return float(auroc), float(auprc)


def evaluate_checkpoint_for_splits(model_state_path: Path) -> dict[str, dict[str, float]]:
    model = build_model()
    state_dict = torch.load(model_state_path, map_location="cpu")
    state_dict = _normalize_state_dict_keys(state_dict)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    with torch.inference_mode():
        train_good_nll = compute_sequence_nll(model, train_good_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        train_bad_nll = compute_sequence_nll(model, train_bad_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_good_nll = compute_sequence_nll(model, val_good_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_bad_nll = compute_sequence_nll(model, val_bad_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_1_good_nll = compute_sequence_nll(model, val_1_good_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_1_bad_nll = compute_sequence_nll(model, val_1_bad_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()

    return {
        "train": dict(zip(["auroc", "auprc"], compute_auroc_auprc_from_nll(train_good_nll, train_bad_nll))),
        "val": dict(zip(["auroc", "auprc"], compute_auroc_auprc_from_nll(val_good_nll, val_bad_nll))),
        "val_1": dict(zip(["auroc", "auprc"], compute_auroc_auprc_from_nll(val_1_good_nll, val_1_bad_nll))),
    }


# Baseline a iterazione 0: modello pretrained iniziale.
baseline_metrics = evaluate_checkpoint_for_splits(pretrained_ckpt_path)
baseline_metrics

In [ ]:
all_metric_rows: list[dict[str, object]] = []
missing_checkpoint_summary: dict[str, list[int]] = {}


def select_target_iterations(history_iterations: list[int], step: int = 5000) -> list[int]:
    if len(history_iterations) == 0:
        return []

    max_iter = max(history_iterations)
    targets = {
        it for it in history_iterations
        if it == 0 or it == max_iter or (it > 0 and it % step == 0)
    }
    return sorted(targets)


# 1) Baseline iterazione 0 calcolata una sola volta (gia' in baseline_metrics)
#    e replicata su tutti i run che hanno iteration=0 tra le iterazioni target.

for history_path_str in history_paths:
    history_abs = resolve_history_path(history_path_str)
    if not history_abs.exists():
        raise FileNotFoundError(f"History file non trovato: {history_abs}")

    with history_abs.open("r", encoding="utf-8") as f:
        history_obj = json.load(f)

    history_iterations = sorted({int(it) for it in history_obj.get("iteration", [])})
    target_iterations = select_target_iterations(history_iterations)
    run_name = history_abs.parent.name

    if 0 in target_iterations:
        for split_name in ["train", "val", "val_1"]:
            all_metric_rows.append(
                {
                    "run": run_name,
                    "history_path": str(history_abs),
                    "checkpoint_dir": str(checkpoint_dir_from_history_path(history_abs)),
                    "iteration": 0,
                    "split": split_name,
                    "auroc": baseline_metrics[split_name]["auroc"],
                    "auprc": baseline_metrics[split_name]["auprc"],
                }
            )

# 2) Per ogni run valuta tutti i checkpoint target: multipli di 5000 e ultima iterazione.
for history_path_str in history_paths:
    history_abs = resolve_history_path(history_path_str)
    if not history_abs.exists():
        raise FileNotFoundError(f"History file non trovato: {history_abs}")

    with history_abs.open("r", encoding="utf-8") as f:
        history_obj = json.load(f)

    history_iterations = sorted({int(it) for it in history_obj.get("iteration", [])})
    target_iterations = [it for it in select_target_iterations(history_iterations) if it != 0]
    if len(target_iterations) == 0:
        continue

    ckpt_dir = checkpoint_dir_from_history_path(history_abs)
    ckpt_by_iter = list_checkpoint_iterations(ckpt_dir)
    run_name = history_abs.parent.name
    missing_iters: list[int] = []

    for target_iter in target_iterations:
        ckpt_path = ckpt_by_iter.get(target_iter)
        if ckpt_path is None:
            missing_iters.append(target_iter)
            continue

        split_metrics = evaluate_checkpoint_for_splits(ckpt_path)
        for split_name in ["train", "val", "val_1"]:
            all_metric_rows.append(
                {
                    "run": run_name,
                    "history_path": str(history_abs),
                    "checkpoint_dir": str(ckpt_dir),
                    "iteration": target_iter,
                    "split": split_name,
                    "auroc": split_metrics[split_name]["auroc"],
                    "auprc": split_metrics[split_name]["auprc"],
                }
            )

    if len(missing_iters) > 0:
        missing_checkpoint_summary[run_name] = missing_iters

auroc_auprc_df = pd.DataFrame(all_metric_rows).sort_values(["run", "iteration", "split"]).reset_index(drop=True)

print(f"Totale righe metriche calcolate: {len(auroc_auprc_df)}")
if len(missing_checkpoint_summary) > 0:
    print("Iterazioni target senza checkpoint corrispondente:")
    for run_name, miss in missing_checkpoint_summary.items():
        print(f"  {run_name}: {miss}")

auroc_auprc_df.head(12)

In [ ]:
if "auroc_auprc_df" not in globals() or auroc_auprc_df.empty:
    raise RuntimeError("Nessuna metrica disponibile. Esegui prima la cella precedente.")

splits = ["train", "val", "val_1"]
fig, axes = plt.subplots(2, 3, figsize=(24, 10), sharex=False, sharey=False)

for col_idx, split_name in enumerate(splits):
    ax = axes[0, col_idx]
    sub = auroc_auprc_df[auroc_auprc_df["split"] == split_name]
    for run_name, run_df in sub.groupby("run"):
        ax.plot(run_df["iteration"], run_df["auroc"], linewidth=2, marker="o", markersize=3, label=run_name)
    ax.set_title(f"AUROC - {split_name}")
    ax.set_xlabel("iteration")
    ax.set_ylabel("AUROC")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

for col_idx, split_name in enumerate(splits):
    ax = axes[1, col_idx]
    sub = auroc_auprc_df[auroc_auprc_df["split"] == split_name]
    for run_name, run_df in sub.groupby("run"):
        ax.plot(run_df["iteration"], run_df["auprc"], linewidth=2, marker="o", markersize=3, label=run_name)
    ax.set_title(f"AUPRC - {split_name}")
    ax.set_xlabel("iteration")
    ax.set_ylabel("AUPRC")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle("Confronto checkpoint per iterazione: AUROC/AUPRC su train, val, val_1", fontsize=15)
plt.tight_layout()
plt.show()